In [69]:
import pandas as pd
DATA_PATH = 'jobpostingdata.csv'
MODEL_PATH = 'model.pkl'
df = pd.read_csv(DATA_PATH)
df.head()

,title,fraudulent,text
0,PHP Developer,0,PHP Developer You're a skilled developer. You ...
1,CUSTOMER SERVICE AGENT,1,CUSTOMER SERVICE AGENT Aegis is a global busi...
2,VP Marketing & Growth,0,VP Marketing & Growth Depop is an exciting new...
3,SAP BW Developer/Architect,1,SAP BW Developer/Architect Assist with Full L...
4,Administrative Assistant,1,Administrative Assistant With decades of exper...


In [70]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [71]:
def get_tag(tag):
    if tag[0] in ['V', 'N', 'R']:
        return tag[0].lower()
    elif tag[0] == 'J':
        return 'a'
    else:
        return 'n'
    
def lemmatizing(tokens):
    lemmatizer = WordNetLemmatizer()
    tagged = pos_tag(tokens)
    lemmatized = []
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, get_tag(tag))
        lemmatized.append(result)
    return lemmatized

def text_preprocessor(sentence):
    eng_stopwords = stopwords.words('english')
    tokens = word_tokenize(sentence)
    cleaned = [
        token.lower() for token in tokens if token not in eng_stopwords
        and
        token.isalpha()
    ]
    lemmatized = lemmatizing(cleaned)
    return lemmatized

In [72]:
def feature_extraction(sentence, existed_words):
    unique_words = set(text_preprocessor(sentence))
    feature = {}
    for word in existed_words:
        feature[word] = (word in unique_words)
    return feature

In [73]:
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.classify import accuracy
from sklearn.model_selection import train_test_split

In [74]:
def train_classifier():
    x = df['text']
    y = df['fraudulent']
    corpus = ' '.join(x)
    existed_words = set(text_preprocessor(corpus))

    features = [
        (feature_extraction(sentence, existed_words), category)
        for sentence, category in zip(x, y)
    ]

    train_data, test_data = train_test_split(
        features, test_size=0.2, random_state=42
    )

    classifier = NaiveBayesClassifier.train(train_data)
    acc_perc = accuracy(classifier, test_data) * 100
    print(f"Accuracy: {acc_perc:.2f}%")

    return classifier, existed_words

In [75]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [76]:
def train_vectorizer():
    x = df['text']
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english'
    )
    vectors = vectorizer.fit_transform(x)
    return vectorizer, vectors

In [77]:
import os
import pickle

In [78]:
def load_model():
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, 'rb') as f:
            classifier, existed_words, vectorizer, vectors = pickle.load(f)
        return classifier, existed_words, vectorizer, vectors
    else:
        classifier, existed_words = train_classifier()
        vectorizer, vectors = train_vectorizer()

        with open(MODEL_PATH, 'wb') as f:
            pickle.dump((classifier, existed_words, vectorizer, vectors), f)
        return classifier, existed_words, vectorizer, vectors

In [79]:
def print_menu(sentence, category):
    print(f"Sentence: {sentence}")
    print(f"Category: {category}")
    print("1. Write Sentence")
    print("2. Recommendation")
    print("3. NER")
    print("4. Exit")

In [80]:
def write_sentence():
    sent = ''
    while len(sent.strip()) < 20 or len(sent.strip().split()) < 3:
        sent = input("Input a sentence: ")
    return sent

In [81]:
def classify_text(classifier, sentence, existed_words):
    feature = feature_extraction(sentence, existed_words)
    category = classifier.classify(feature)
    if category == 1:
        return 'FRAUD NI-'
    else:
        return 'TRUE G'

In [82]:
def load_NER(nlp, sentence):
    doc = nlp(sentence)
    ents_dicts = {}

    for ent in doc.ents:
        if ent.label_ not in ents_dicts.keys():
            ents_dicts[ent.label_] = []
        ents_dicts[ent.label_].append(ent.text)
    return ents_dicts

In [83]:
def print_NER(ents_dicts):
    if ents_dicts:
        for key, value in ents_dicts.items():
            print(f"{key}: {value}")
    else:
        print(f"No N E R")

In [84]:
from sklearn.metrics.pairwise import cosine_similarity

In [85]:
def recommend_top_n(vectorizer: TfidfVectorizer, job_vectors, query, topn=5):
    vectorized_query = vectorizer.transform([query])
    similarity = cosine_similarity(job_vectors, vectorized_query).flatten()
    topidx = similarity.argsort()[::-1][:topn]

    for i, idx in enumerate(topidx, 1):
        print(f"{i}. {df['title'].iloc[idx]}")

In [86]:
import spacy

In [87]:
def menu():
    sent = ''
    cat = ''

    classifier = None
    existed_words = None
    vectorizer = None
    vectors = None
    ents_dicts = {}
    nlp = spacy.load("en_core_web_sm")

    while True:
        print()
        print_menu(sent, cat)
        choice = input("Select your choice: ")

        if choice == '1':
            sent = write_sentence()

            if classifier is None or existed_words is None or vectorizer is None or vectors is None:
                classifier, existed_words, vectorizer, vectors = load_model()
            cat = classify_text(classifier, sent, existed_words)

        elif choice == '2':
            if sent == '':
                print("No Sentence found")
                continue

            recommend_top_n(vectorizer, vectors, sent)
        
        elif choice == '3':
            if sent == '':
                print("No Sentence found")
                continue

            ents_dicts = load_NER(nlp, sent)
            print_NER(ents_dicts)
        
        elif choice == '4':
            break

In [88]:
menu()


Sentence: 
Category: 
1. Write Sentence
2. Recommendation
3. NER
4. Exit
Accuracy: 75.00%

Sentence: i have a IT job tomorrow at Apple
Category: FRAUD NI-
1. Write Sentence
2. Recommendation
3. NER
4. Exit
1. Part Time Job Work From Home, Daily Pay.
2. Recruiting Coordinator
3. iOS Developer
4. Frac Supervisor
5. Head of Partnerships

Sentence: i have a IT job tomorrow at Apple
Category: FRAUD NI-
1. Write Sentence
2. Recommendation
3. NER
4. Exit
DATE: ['tomorrow']
ORG: ['Apple']

Sentence: i have a IT job tomorrow at Apple
Category: FRAUD NI-
1. Write Sentence
2. Recommendation
3. NER
4. Exit
DATE: ['tomorrow']
ORG: ['Apple']

Sentence: i have a IT job tomorrow at Apple
Category: FRAUD NI-
1. Write Sentence
2. Recommendation
3. NER
4. Exit
